# Kotoba STT - Real-time Speech-to-Text with Bidirectional Streaming

This notebook shows how to deploy and call the **Kotoba STT** model package from AWS Marketplace, end-to-end.

**Product**: Kotoba STT - Real-time Speech Transcription
**Seller**: Kotoba AI

## Key Features

- Real-time streaming transcription with sub-second latency
- Japanese support
- Multiple audio formats (`pcm16`, `float32`, `mulaw`, `opus`)
- Scalable GPU-accelerated inference (NVIDIA L4)

---

## IMPORTANT: Bidirectional streaming only

Kotoba STT uses **SageMaker bidirectional streaming exclusively**. It is **not** a standard REST endpoint:

- `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]`

If these constraints don't fit your workload, see the Troubleshooting section at the end.

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba STT

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Inference](#4.-Perform-Real-time-Inference)
5. [Troubleshooting](#5.-Troubleshooting)
6. [Clean-up](#6.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the Kotoba STT model package:

1. Open the [Kotoba STT listing on AWS Marketplace](#) <!-- TODO: Update with actual URL -->
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first — the Python version check in 2.2 doesn't require these packages, but every cell after that does.

In [ ]:
!pip install -q --upgrade boto3 numpy soundfile
!pip install -q --upgrade "sagemaker>=2.0,<3"
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]"


### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")


### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
from pathlib import Path

import boto3
import numpy as np
import sagemaker
import soundfile as sf

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import StaticCredentialsResolver

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` will pick up the attached role automatically.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.


In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")


### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )


## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

Supported instance types:

| Instance | vCPU | Memory | GPU | Notes |
|----------|------|--------|-----|-------|
| `ml.g6.2xlarge` | 8  | 32 GiB | 1x NVIDIA L4 | Default, good balance of cost and throughput |
| `ml.g6.4xlarge` | 16 | 64 GiB | 1x NVIDIA L4 | Choose for higher concurrency |


In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-stt-{int(time.time())}"
INSTANCE_TYPE = "ml.g6.2xlarge"  # or "ml.g6.4xlarge"

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")


In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes 5-10 minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")


## 4. Perform Real-time Inference

### 4.1 Connection Protocol

Kotoba STT is reached through SageMaker's bidirectional-streaming transport:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Container
```

Standard WebSocket libraries **cannot** connect to SageMaker endpoints directly — you must use `aws-sdk-python[sagemaker-runtime-http2]`.

### 4.2 Audio Format

| Format | Sample rates (Hz) | Channels | Encoding |
|--------|-------------------|----------|----------|
| `pcm16`   | 24000 (Recommended) | 1 (mono) | Little-endian int16 |
| `float32` | 24000 (Recommended) | 1 (mono) | Little-endian float32, range `[-1.0, 1.0]` |
| `mulaw`   | 8000 (μ-law)               | 1 (mono) | G.711 μ-law  |
| `opus`    | 24000, 48000               | 1 (mono) | Self-contained Ogg/Opus per `append` |

- Default sample rate: **24000 Hz**. Other rates are resampled server-side.
- Recommended chunk size: **80 ms** per `input_audio_buffer.append` event.
- Maximum payload per `input_audio_buffer.append`: **~1 MiB (~8 seconds at 24 kHz pcm16)**.

### 4.3 Protocol Flow

```
Client                                         Server
  |<----- transcription_session.created -------|
  |------ transcription_session.update ------->|
  |<----- transcription_session.updated -------|
  |<----- conversation.item.created -----------|   (emitted once, before first delta)
  |------ input_audio_buffer.append ---------->|   (repeated, streaming)
  |<----- conversation.item.input_audio_transcription.delta --|   (repeated)
  |------ input_audio_buffer.commit ---------->|
  |<----- input_audio_buffer.committed --------|
```

### 4.4 Event Reference

#### Client Events (you send)

| Event | Purpose |
|-------|---------|
| `transcription_session.update` | Configure the session (audio format, language, optional keywords). Send **once** after `transcription_session.created`. |
| `input_audio_buffer.append`    | Send one audio chunk (base64-encoded bytes). Repeat as audio arrives. |
| `input_audio_buffer.commit`    | Signal "no more audio." Triggers final flush on the server. |

#### Server Events (you receive)

| Event | Purpose |
|-------|---------|
| `transcription_session.created` | Session is ready — send your `transcription_session.update` next. |
| `transcription_session.updated` | Server accepted your config. Safe to start streaming audio. |
| `conversation.item.created`     | A conversation item was opened to hold the incoming transcript. Informational — no action needed. |
| `conversation.item.input_audio_transcription.delta` | Transcription fragment. Concatenate `delta` fields in order. |
| `input_audio_buffer.committed`  | All audio processed. Close the stream after this. |
| `error`                          | Something went wrong. Payload: `{"type": "error", "event_id": ..., "error": {...}, "item_id"?: ...}`. |

> **Note**: `turn_detection` must be `false`. Automatic turn detection (and its accompanying `*.transcription.completed` event) is not currently supported — rely on the concatenation of `delta` events plus `input_audio_buffer.committed` as the "done" signal.

See [docs/stt/data/sample_input.json](./data/sample_input.json) and [docs/stt/data/sample_output.json](./data/sample_output.json) for canonical event payloads.

### 4.5 Prepare Input Audio

Two paths:

- **Smoke test** (next cell): sends 3 seconds of silence. Useful purely to confirm the endpoint is reachable and the protocol round-trips. The transcript will be empty.
- **Real audio** (cell after): loads a WAV file. Use this to see an actual transcription.


In [ ]:
# --- Connectivity smoke test (silent audio) ---------------------------------
# Emits 3 s of silence, then runs the full transcription_session ->
# input_audio_buffer.commit flow. Expect an EMPTY transcript; the goal is to
# confirm the endpoint routes events correctly end-to-end.

SAMPLE_RATE = 24000
CHUNK_DURATION_MS = 80  # recommended chunk size

smoke_test_samples = np.zeros(SAMPLE_RATE * 3, dtype=np.int16)
smoke_test_bytes = smoke_test_samples.tobytes()
SMOKE_TEST_LANGUAGE = "ja"

print(f"Smoke-test audio ready: {len(smoke_test_bytes)} bytes "
      f"({len(smoke_test_samples) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz, pcm16)")


### 4.5b Load Your Own Audio (recommended)

Replace `WAV_PATH` with any audio file readable by [`soundfile`](https://pysoundfile.readthedocs.io/) (WAV / FLAC / OGG / ...). The cell below validates mono + a supported sample rate, decodes to int16, and prepares it for streaming.

For a quick win, you can try the English 10-second sample shipped with this repo:

```python
WAV_PATH = "path/to/audio"
AUDIO_LANGUAGE = "ja"
```


In [ ]:
# ----- Replace THIS with your own audio file -------------------------------
WAV_PATH = "path/to/audio"   # any format supported by libsndfile (WAV, FLAC, OGG, ...)
AUDIO_LANGUAGE = "ja"                   # "ja"
# ---------------------------------------------------------------------------


def load_pcm16(path: str) -> tuple[bytes, int]:
    """Load an audio file and return (int16 little-endian PCM bytes, sample_rate).

    Accepts any format `soundfile` can decode (WAV, FLAC, OGG, etc.). Requires mono.
    """
    data, samplerate = sf.read(path, dtype="int16")
    if data.ndim != 1:
        raise ValueError(
            f"{path}: expected mono, got {data.shape[1]} channels. "
            "Mix down to mono before passing to Kotoba STT."
        )
    if samplerate not in (16000, 24000, 44100, 48000):
        raise ValueError(
            f"{path}: unsupported sample rate {samplerate}. "
            "Supported: 16000, 24000, 44100, 48000 Hz."
        )
    return data.tobytes(), samplerate


if Path(WAV_PATH).exists():
    real_audio_bytes, real_audio_rate = load_pcm16(WAV_PATH)
    print(
        f"Loaded {WAV_PATH}: {len(real_audio_bytes)} bytes "
        f"({len(real_audio_bytes) / 2 / real_audio_rate:.2f} s @ {real_audio_rate} Hz)"
    )
else:
    real_audio_bytes = None
    real_audio_rate = None
    print(
        f"WAV_PATH={WAV_PATH!r} not found. Update the path above or skip this cell "
        "and run the smoke test only."
    )


### 4.6 Streaming Inference

It establishes the HTTP/2 stream, waits for `transcription_session.created` (with a timeout),
sends configuration, streams audio chunks with real-time pacing, and concatenates
`delta` events into the full transcript.

### Session configuration options

The `session` object inside `transcription_session.update` supports:

| Field | Type | Default | Notes |
|-------|------|---------|-------|
| `input_audio_format` | `"pcm16"` \| `"float32"` \| `"mulaw"` \| `"opus"` | `"pcm16"` | Must match the bytes you send. |
| `input_audio_sample_rate` | int | `24000` | Server resamples to 24 kHz internally. |
| `input_audio_number_of_channels` | int | `1` | Mono only. |
| `input_audio_transcription.target_language` | str | *(required)* | `"ja"` |
| `input_audio_transcription.language` | str | `None` | If set, must equal `target_language`. |
| `input_audio_transcription.keywords` | list[str] | `None` | Bias recognition toward the listed terms. |
| `input_audio_transcription.kana` | bool | `False` | Transcribe person names in katakana. |


In [ ]:
SESSION_INIT_TIMEOUT = 10.0   # seconds to wait for transcription_session.created/updated
SESSION_COMMIT_TIMEOUT = 30.0  # seconds to wait for input_audio_buffer.committed after flush
CHUNK_SAMPLES_PCM16 = int(SAMPLE_RATE * CHUNK_DURATION_MS / 1000)
CHUNK_BYTES_PCM16 = CHUNK_SAMPLES_PCM16 * 2  # 2 bytes per int16 sample


async def transcribe(
    endpoint_name: str,
    audio_bytes: bytes,
    *,
    sample_rate: int = SAMPLE_RATE,
    language: str = "ja",
    client_id: str = "nb",
) -> str:
    """Transcribe pcm16 mono audio via Kotoba STT bidirectional streaming."""

    credentials = boto3.Session().get_credentials().get_frozen_credentials()
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        aws_credentials_identity_resolver=StaticCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )
    client = SageMakerRuntimeHTTP2Client(config=config)

    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    # NOTE: data_type="UTF8" is required. Without it the SageMaker runtime
    # relays our payloads as WebSocket *binary* frames, but the server calls
    # starlette's websocket.receive_json() which only accepts text frames
    # (it internally reads message["text"]). The flag tells SageMaker the
    # bytes are UTF-8 JSON so the frame is delivered as text.
    async def send_event(payload: dict) -> None:
        body = json.dumps(payload).encode("utf-8")
        await stream.input_stream.send(
            RequestStreamEventPayloadPart(
                value=RequestPayloadPart(bytes_=body, data_type="UTF8")
            )
        )

    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output

    async def next_event() -> dict:
        message = await output_stream.receive()
        if message is None:
            raise RuntimeError("Stream closed unexpectedly")
        return json.loads(message.value.bytes_.decode("utf-8"))

    # 1) Wait for transcription_session.created
    async with asyncio.timeout(SESSION_INIT_TIMEOUT):
        event = await next_event()
        if event["type"] == "error":
            raise RuntimeError(f"Server error during init: {event}")
        assert event["type"] == "transcription_session.created", event
        print("[session] created")

        # 2) Send session configuration
        await send_event(
            {
                "type": "transcription_session.update",
                "session": {
                    "input_audio_format": "pcm16",
                    "input_audio_sample_rate": sample_rate,
                    "input_audio_number_of_channels": 1,
                    "input_audio_transcription": {
                        "language": language,
                        "target_language": language,
                        "keywords": [],
                        "kana": False,
                    },
                },
            }
        )

        # 3) Wait for transcription_session.updated
        event = await next_event()
        if event["type"] == "error":
            raise RuntimeError(f"Server error during update: {event}")
        assert event["type"] == "transcription_session.updated", event
        print("[session] updated")

    # 4) Stream audio chunks (sender) while collecting deltas (receiver)
    transcript_parts: list[str] = []
    chunk_bytes = int(sample_rate * CHUNK_DURATION_MS / 1000) * 2  # pcm16 = 2 bytes/sample

    async def send_audio() -> None:
        for i in range(0, len(audio_bytes), chunk_bytes):
            chunk = audio_bytes[i : i + chunk_bytes]
            await send_event(
                {
                    "event_id": f"{client_id}_{i // chunk_bytes:07d}",
                    "type": "input_audio_buffer.append",
                    "audio": base64.b64encode(chunk).decode(),
                }
            )
            await asyncio.sleep(CHUNK_DURATION_MS / 1000)  # simulate real-time pacing
        await send_event({"type": "input_audio_buffer.commit"})

    async def receive_deltas() -> None:
        async with asyncio.timeout(SESSION_COMMIT_TIMEOUT):
            while True:
                event = await next_event()
                etype = event["type"]
                if etype == "conversation.item.input_audio_transcription.delta":
                    delta = event.get("delta", "")
                    transcript_parts.append(delta)
                    print(delta, end="", flush=True)
                elif etype == "conversation.item.created":
                    continue  # informational
                elif etype == "input_audio_buffer.committed":
                    print()  # newline after streaming deltas
                    return
                elif etype == "error":
                    raise RuntimeError(f"Server error: {event}")

    await asyncio.gather(send_audio(), receive_deltas())
    await stream.input_stream.close()

    return "".join(transcript_parts)


### 4.7 Run Transcription

**Smoke test first** — confirms the endpoint is healthy. Expect an empty transcript.

In [ ]:
smoke_transcript = await transcribe(
    endpoint_name=ENDPOINT_NAME,
    audio_bytes=smoke_test_bytes,
    sample_rate=SAMPLE_RATE,
    language=SMOKE_TEST_LANGUAGE,
    client_id="smoke",
)
print(f"\nSmoke-test transcript (expected empty): {smoke_transcript!r}")


**Now the real audio** — run this only after the smoke test above completes successfully.

In [ ]:
if real_audio_bytes is None:
    print("Skipping: real audio not loaded. Set WAV_PATH in the loader cell and re-run.")
else:
    transcript = await transcribe(
        endpoint_name=ENDPOINT_NAME,
        audio_bytes=real_audio_bytes,
        sample_rate=real_audio_rate,
        language=AUDIO_LANGUAGE,
        client_id="nb",
    )
    print(f"\n--- Full transcript ---\n{transcript}")


## 5. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | Your kernel is < 3.12. Create a new environment with 3.12+ and re-attach the kernel. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | The HTTP/2 SDK install failed. Re-run cell 2.1. On SageMaker Studio, make sure the kernel is restarted after install. |
| `TimeoutError` waiting for `transcription_session.created` | Endpoint is not yet `InService`, the endpoint name is wrong, or the region in `boto3.Session()` doesn't match the endpoint's region. Verify with `sagemaker_session.describe_endpoint(EndpointName=ENDPOINT_NAME)`. |
| `TimeoutError` waiting for `input_audio_buffer.committed` | Model is overloaded, or the posted audio is too long to process within `SESSION_COMMIT_TIMEOUT`. Raise the timeout or split long audio into shorter sessions. |
| `Server error: {'type': 'error', 'error': {...}}` | Server-side validation failure. Common causes: `target_language` not `"ja"`, append payload exceeds ~1 MiB. |
| Transcript is empty on real audio | Double-check `AUDIO_LANGUAGE` matches the spoken language, confirm the WAV is mono 16-bit PCM at a supported rate, and verify the audio is actually audible (not silence). |
| `AccessDeniedException` at deploy time | Your IAM role lacks the permissions listed in section "Required IAM Permissions" above, or you haven't completed the AWS Marketplace subscription. |


## 6. Clean-up

**Important**: Delete the endpoint when done. GPU instances (`ml.g6.2xlarge` / `ml.g6.4xlarge`) are billed **per hour while the endpoint is active**, even when idle.

In [ ]:
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)

print("Endpoint deleted.")


### Unsubscribe from Model Package (Optional)

To unsubscribe from the Kotoba STT model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba STT" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, please contact Kotoba AI support.

## References

- Canonical protocol samples: [docs/stt/data/sample_input.json](./data/sample_input.json), [docs/stt/data/sample_output.json](./data/sample_output.json), [docs/stt/data/README.md](./data/README.md)
- Production reference client: [bench/kotoba_bench/sagemaker_http2.py](../../bench/kotoba_bench/sagemaker_http2.py)
- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)